# Module 7, Section 2: Insights

Section 1 found the wrong return window because you knew what to look for and wrote a check for it. An online evaluator answers a question you write down in advance. Insights finds the questions you had not thought to ask.

**Insights** is a LangSmith feature that reads a batch of traces, groups them into categories and subcategories, and writes a report with statistics for each group.

In this section:
1. How Insights builds a report
2. Reading a report top down: summary, categories, subcategories
3. Silent failures have a 0% error rate: checking one claim against the traces
4. Deciding what each finding should become
5. From a subcategory to an annotation queue to a dataset
6. Running a real report in the LangSmith UI

## Setup

This section calls no model and needs no key. It reads a sample report that ships with the module, `workshop_modules/module_7/data/insights_report_techhub.json`. The sample is hand-built from the seed, so every number in it matches the traces, and it has the documented structure of a real report.

A real report has three requirements:

| Requirement | Detail |
|---|---|
| Plan | LangSmith Plus or Enterprise. A free Developer workspace cannot run Insights |
| Model | A model configuration for Insights, backed by a provider key stored as a workspace secret. The report's model calls are billed to that key |
| Time | Up to 30 minutes per report |

| Where you are | What to set |
|---|---|
| No LangSmith key, or no access to Insights | Nothing. The notebook reads the sample report |
| A key and a finished report on your TechHub traces | `M7_INSIGHTS_REPORT_ID` in `.env`: the report's id, which is the `clusterJobId` value in the report's URL. The report must belong to the project that the Setup cell names after "Seeded traffic goes to:" |

The checks always grade the sample report, because the answer key describes the seeded day. One cell can write to your workspace, and only when you set its flag: `CREATE_DATASET` in §5 adds one test case to a dataset.

In [ ]:
import json, os, sys, warnings
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(REPO_ROOT))
# Load .env before any workshop import: config.py picks the models from it at import time
load_dotenv(REPO_ROOT / ".env")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=r".*is deprecated and will be removed after", category=DeprecationWarning)

from utils.langsmith_links import setup_langsmith
from utils.trace_seeding import describe, load_seed, print_trace
from utils.trace_upload import resolve_destination

LANGSMITH_AVAILABLE = setup_langsmith()            # one probe at most, one printed line
DEST = resolve_destination(LANGSMITH_AVAILABLE)    # project name, project id, navigation sentence
DATA = REPO_ROOT / "workshop_modules" / "module_7" / "data"
seed = load_seed()                                 # timestamps shifted so the newest run is 5 minutes old

## 1. How Insights works

Insights builds a report in three steps. The numbers on the left are those of the TechHub day:

```text
260 traces in 200 conversations
   │  1. Summarize: a cheaper model (the Summarization model) writes a short summary of each conversation
   ▼
200 summaries
   │  2. Cluster: a more capable model (the Thinking model) groups the summaries into categories and subcategories
   ▼
4 categories, 10 subcategories
   │  3. Report: an executive summary with key points and highlighted traces, and statistics for each group
   ▼
the report
```

Only what is in a summary can influence the categories. You can name the top-level categories yourself; the subcategories are always generated.

| Fact | Value |
|---|---|
| What it reads | Traces (root runs), at most 1,000 per report. In a project with threads it analyses whole conversations, and only the most recent root run of each thread fills the `run.*` variables of the summary prompt |
| How long it takes | Up to 30 minutes, according to the docs. No time is documented for a small set of traces |
| Who pays | The workspace's own provider key. The docs give about $1.00 to $2.00 per 1,000 threads with OpenAI models and $3.00 to $4.00 with Anthropic models |
| Who can do it | Viewing a report needs read access to the project. Starting one needs permission to create rules, which Editors and Admins have |

The TechHub traces have threads, so the sample report counts conversations: its groups add up to 200.

In [ ]:
from langsmith.schemas import InsightsReportResult
from utils.trace_seeding import load_insights_report
from utils.trace_upload import new_client

INSIGHTS_REPORT_ID = os.getenv("M7_INSIGHTS_REPORT_ID")  # the id of a finished report on your TechHub traces (optional)

# The sample is a dictionary in the REST shape of a report, with its times moved along with the seed.
# It loads into the same type that client.get_insights_report() returns.
sample = InsightsReportResult(**load_insights_report(seed))
report = sample
if LANGSMITH_AVAILABLE and INSIGHTS_REPORT_ID:
    if DEST.project_id is None:
        print(f"⏭️  The project {DEST.project_name} has no traces yet, so report {INSIGHTS_REPORT_ID} cannot be read there.")
    else:
        try:
            report = new_client().get_insights_report(id=INSIGHTS_REPORT_ID, project_id=DEST.project_id, include_runs=False)
            print(f"Reading your report {INSIGHTS_REPORT_ID}.")
        except Exception as exc:  # noqa: BLE001 - any failure falls back to the sample report
            print(f"⏭️  Could not read report {INSIGHTS_REPORT_ID} ({type(exc).__name__}).")
if report is sample:
    print("⏭️  Reading the sample report shipped with the module (hand-built from the seed, not real Insights output).")

print()
print(report.name)
print(f"Status: {report.status}. Finished: {report.end_time}.")
print(f"report.clusters holds {len(list(report.clusters))} groups in one flat list: the categories and their subcategories.")

In [ ]:
from utils.trace_seeding import root_runs, trace_key

summary = report.report
keys_by_run_id = {run["id"]: trace_key(run) for run in root_runs(seed)}  # run id -> seed trace key such as t-0169

print(summary.title)
print()
print("Key points")
for number, point in enumerate(summary.key_points, start=1):
    print(f"{number}. {point}")
print()
print("Highlighted traces")
for trace in sorted(summary.highlighted_traces, key=lambda t: t.rank):
    where = keys_by_run_id.get(str(trace.run_id), str(trace.run_id))
    print(f"#{trace.rank}  {trace.cluster_name}  (trace {where})")
    print(f"    {trace.highlight_reason}")

## 2. Read it top down

Read a report in the order the UI shows it: the executive summary, the categories, the subcategories, then the traces behind them.

| Look at | What to look for |
|---|---|
| Executive summary | The claims the report makes, with their percentages. Each highlighted trace (#1, #2, ...) is an example the report picked for a claim |
| Categories | Where the traffic is: the share of conversations in each category |
| Subcategories | The outliers: the highest error rate, the highest cost, and a description that names a failure while the error rate is 0 |
| Traces | The evidence. Open a highlighted trace before you act on a claim |

Each group has a `stats` dictionary. This module uses three of its keys: `error_rate` (a fraction from 0 to 1), `latency_p50` (the median latency in seconds) and `cost_p50` (the median cost in dollars). In the sample report, `error_rate` is the share of conversations with an errored run. A real report also shows, per group, the feedback scores of your evaluators and any attributes you asked for; the docs do not name their keys.

A cluster whose `parent_id` is `None` is a top-level category, and every other cluster is a subcategory. The meaning of the `level` number is not documented, so the code below uses `parent_id`, which the REST schema defines.

> 💡 **Try it first:** write `summarise(report)` in `workshop_modules/module_7/exercises/read_the_report.py`. It receives the report as a dictionary, as `utils.trace_seeding.load_insights_report()` returns it, and returns the largest category and the subcategory with the highest error rate. Check it from the repository root with `uv run python workshop_modules/module_7/exercises/check.py s2_report`. To read the whole report in a terminal first, run `uv run python workshop_modules/module_7/exercises/show_report.py`.

In [ ]:
def stats_of(cluster):
    return cluster.stats or {}  # the schema allows stats to be None


def row(name, cluster, total):
    stats = stats_of(cluster)
    cost = f"${stats.get('cost_p50', 0.0):.4f}"
    return (f"{name:<36} {cluster.num_runs:>5} {cluster.num_runs / total:>6.0%} {stats.get('error_rate', 0.0):>7.1%} "
            f"{stats.get('latency_p50', 0.0):>7.1f}s {cost:>9}")


clusters = list(report.clusters)
categories = sorted((c for c in clusters if c.parent_id is None), key=lambda c: -c.num_runs)
total = sum(c.num_runs for c in categories)  # every conversation is in exactly one top-level category

print(f"{'Category':<36} {'convs':>5} {'share':>6} {'errors':>7} {'latency':>8} {'cost':>9}")
for category in categories:
    print(row(category.name, category, total))
print()
print(f"{total} conversations. 'errors' is the error rate; latency and cost are medians (p50).")

In [ ]:
from utils.monitoring_checks import check


def summary_of(report):
    """The two headline facts of a report: its largest category and its subcategory with the highest error rate."""
    clusters = list(report.clusters)
    tops = [c for c in clusters if c.parent_id is None]
    subs = [c for c in clusters if c.parent_id is not None]
    largest = max(tops, key=lambda c: c.num_runs)
    worst = max(subs, key=lambda c: stats_of(c).get("error_rate", 0.0))
    return {
        "largest_category": largest.name,
        "largest_category_conversations": largest.num_runs,
        "largest_category_share": round(largest.num_runs / sum(c.num_runs for c in tops), 2),
        "highest_error_rate_subcategory": worst.name,
        "highest_error_rate": stats_of(worst)["error_rate"],
    }


print(f"{'Category and subcategory':<36} {'convs':>5} {'share':>6} {'errors':>7} {'latency':>8} {'cost':>9}")
for category in categories:
    print(row(category.name, category, total))
    children = sorted((c for c in clusters if c.parent_id == category.id), key=lambda c: -c.num_runs)
    for child in children:
        print(row("  " + child.name, child, total))

headline = summary_of(report)
print()
print(f"Largest category: {headline['largest_category']} ({headline['largest_category_conversations']} conversations, {headline['largest_category_share']:.0%}).")
print(f"Highest error rate: {headline['highest_error_rate_subcategory']} ({headline['highest_error_rate']:.1%}).")
if report is not sample:
    print("The check below grades the sample report, because the answer key describes the seeded day.")
ok, message = check("s2_report", summary_of(sample))
print(message)

## 3. Silent failures have a 0% error rate

Two subcategories show an error rate of 0.0%: "Return window questions" and "Order reported as not found". Nothing crashed in either, so an error-rate chart shows nothing for them. The report found them from what the conversations say.

The category names, the descriptions and the key points are model output. The metadata, the tool results and the replies in the traces are what happened. Check a claim against the traces before you act on it. The next cell checks the report's claim about return windows with the evaluator from Section 1.

The cell takes the 27 conversations of the subcategory from `seed_labels.json`. That label file exists only because the data is synthetic. With a real report you get the same list from the subcategory's traces table in the UI, or from `report.clusters["Return window questions"].load_traces()` on a report read with `client.get_insights_report`. The keys of the runs that call returns are not documented (UNVERIFIED).

> 💡 **Try it first:** write `flagged_in_subcategory(name, seed, labels, perform_eval)` in the same file, `read_the_report.py`. It counts the conversations of one subcategory that an evaluator flags. Check it with `uv run python workshop_modules/module_7/exercises/check.py s2_claim`.

In [ ]:
from collections import Counter
from evaluators import wrong_return_window_evaluator
from utils.monitoring_local import apply_evaluator, flagged_trace_keys
from utils.trace_seeding import load_labels, trace_runs

CLAIMED = "Return window questions"
# This cell reads the sample report even when your own report is loaded: the label file describes the seeded day.
claim = next(point for point in sample.report.key_points if CLAIMED in point)
cluster = next(c for c in sample.clusters if c.name == CLAIMED)

labels = load_labels()  # ground truth per conversation; it exists only because the data is synthetic
conversations = [c for c in labels["conversations"] if c["subcategory"] == CLAIMED]
roots_by_key = {trace_key(run): run for run in root_runs(seed)}
turns = [roots_by_key[key] for c in conversations for key in c["trace_keys"]]
flagged_keys = set(flagged_trace_keys(apply_evaluator(wrong_return_window_evaluator, turns)))
flagged = [c for c in conversations if flagged_keys & set(c["trace_keys"])]  # a conversation counts once

print("Report:", claim)
print(f"Report: {cluster.num_runs} conversations, error rate {stats_of(cluster)['error_rate']:.1%}.")
print(f"Traces: the Section 1 evaluator flags {len(flagged)} of {len(conversations)} conversations, "
      f"all on app_version {', '.join(sorted({c['app_version'] for c in flagged}))}.")
print()
print("What the 27 conversations were told, by product condition and version:")
answers = Counter((c["details"]["item_condition"], c["app_version"], c["details"]["answered_window_days"]) for c in conversations)
for (condition, version, days), count in sorted(answers.items()):
    right = 30 if condition == "unopened" else 14
    print(f"  {condition:<9} on {version}: {count:>2} told {days} days ({'right' if days == right else 'wrong'})")

top = min(sample.report.highlighted_traces, key=lambda t: t.rank)
evidence = trace_runs(seed, str(top.run_id))
names = {run["name"] for run in evidence}
print()
print(f"Highlighted trace #{top.rank}: {top.highlight_reason}")
print_trace(seed, str(top.run_id))
if "product-policy-specialist" in names and "search_policy_docs" not in names:
    print("product-policy-specialist answered without calling search_policy_docs.")
print()
ok, message = check("s2_claim", {
    "conversations": cluster.num_runs,
    "error_rate": stats_of(cluster)["error_rate"],
    "flagged": len(flagged),
})
print(message)

## 4. What should each finding become?

A finding can go to one of five places:

| Choice | Fits a finding that |
|---|---|
| Alert | Shows in a metric LangSmith already records (errors, latency, cost, a feedback score) and needs someone to act within minutes |
| Online evaluator | Can be written down as a check on one trace, so every new trace can be scored |
| Annotation queue | Needs a person to read the traces and correct them |
| Engine issue | Needs a change to the agent, with a diagnosis and evidence, tracked until it is fixed |
| Nothing | Is the agent behaving as intended |

> 💡 **Try it first:** for each of the five key points of the report, choose one of the five. Some have two defensible answers. The next cell gives a worked answer; it is not graded.

| Key point | Your choice |
|---|---|
| 1. Errors in "Where is my order" after an order-service outage | |
| 2. 30 days given for opened electronics, with a 0.0% error rate | |
| 3. Orders reported as not found, with a 0.0% error rate | |
| 4. Compatibility questions cost about twice as much as spec lookups | |
| 5. Out-of-scope requests refused | |

In [ ]:
worked_answer = [
    ("Errors in 'Where is my order' after an order-service outage",
     "Alert",
     "An error count is a metric LangSmith already records. Section 1 found the threshold: 3 failed traces (errored root runs) in 5 minutes. "
     "The errors came from get_order_status timing out, so the first job is to tell someone while it happens.",
     None),
    ("30 days given for opened electronics, with a 0.0% error rate",
     "Online evaluator",
     "The answers raise no error and got 9 thumbs-up and no thumbs-down, so only a check that knows the policy finds new cases. "
     "Section 1 wrote that check and routed its flagged runs to an annotation queue.",
     "An Engine issue as well, to track the fix to the policy specialist's instructions. That is issue 1 in Section 3."),
    ("Orders reported as not found, with a 0.0% error rate",
     "Engine issue",
     "The agent needs a code change: normalise an order id typed in lower case or without dashes to the ORD-YYYY-NNNN form before the lookup. "
     "An Engine issue holds the diagnosis, the evidence and the pull request. That is issue 2 in Section 3, status fixing.",
     "An online evaluator that flags a reply saying an order does not exist after an 'Order ... not found.' tool result."),
    ("Compatibility questions cost about twice as much as spec lookups",
     "Engine issue, kept as a finding to watch",
     "The answers are correct, so only cost is affected. Engine records it as a finding and counts new cases while it is "
     "watched. That is issue 3 in Section 3.",
     "An alert on Cost, if the spend grows."),
    ("Out-of-scope requests refused",
     "Nothing",
     "The scope guard refused all 11 requests for another customer's data, as designed. The 15 unrelated requests need no change.",
     "An annotation queue with a small sample, if you want people to confirm the refusals from time to time."),
]
for number, (finding, choice, why, also) in enumerate(worked_answer, start=1):
    print(f"{number}. {finding}")
    print(f"   Choice: {choice}")
    print(f"   Why: {why}")
    if also:
        print(f"   Also defensible: {also}")
    print()

## 5. From a subcategory to a queue to a dataset

A subcategory with a real failure ends up as test cases. The Academy recommends review first: people confirm in an annotation queue that the pattern is real and correct each answer, and only then add the traces to a dataset.

> In LangSmith, open **Tracing**. If the control at the top left shows your workspace name (an agent-based workspace), select the agent **TechHub support** in the agent switcher in the top bar, then set the environment control to **Production**. If it shows the LangSmith logo (a project-based workspace), open the tracing project `techhub-deep-agent-production` (or the name you set in `M7_PROJECT`). Set the time range to at least the last 2 days.

| Step | In the UI |
|---|---|
| 1. Open your TechHub traces | **Tracing** > agent **TechHub support** > **Production**; in a project-based workspace, **Tracing** > project `techhub-deep-agent-production`. Set the time range to at least the last 2 days |
| 2. Open the report | The **Insights** tab, then the report |
| 3. Drill down | Click a category, then a subcategory (in the sample: **Returns and refunds**, then **Return window questions**). This opens the subcategory's traces table |
| 4. Select traces | Tick rows with the row checkboxes. One action adds at most 100 items |
| 5. Send them to review | **Add to Annotation Queue** at the bottom of the page. Under **What to add**, choose **Selected run** or **Entire thread**, then the queue `techhub-deep-agent-return-window-review` |
| 6. Review | In the queue, correct the answer to 14 days, then **Add to Dataset** > `techhub-deep-agent-regressions`, as at the end of Section 1 |

The docs describe steps 4 and 5 for any traces table. Whether the report view has its own control that adds a whole subcategory at once is UNVERIFIED.

The next cell builds the same test case in code, from highlighted trace #1: the customer's question and the corrected answer.

In [ ]:
from utils.monitoring_checks import CORRECTED_ANSWER
from utils.trace_seeding import question_text, reply_text

DATASET_NAME = "techhub-deep-agent-regressions"
root = evidence[0]  # the root run of highlighted trace #1, from §3
key = trace_key(root)
example = {
    "inputs": {"messages": [{"role": "user", "content": question_text(root)}]},
    "outputs": {"messages": [{"role": "assistant", "content": CORRECTED_ANSWER}]},
    "metadata": {"seed_trace_key": key, "subcategory": CLAIMED, "source": "module 7 section 2"},
}
print(f"Trace {key}. The agent answered: {reply_text(root)}")
print()
print("The test case:")
print(json.dumps(example, indent=2))
print()

CREATE_DATASET = False  # set to True (with a key) to add this example to the dataset
if CREATE_DATASET and LANGSMITH_AVAILABLE:
    try:
        client = new_client()
        if not client.has_dataset(dataset_name=DATASET_NAME):
            client.create_dataset(DATASET_NAME, description="TechHub deep agent: corrected answers from production traces (Module 7).")
        # Re-running adds nothing: an example built from the same seed trace is already there.
        if next(iter(client.list_examples(dataset_name=DATASET_NAME, metadata={"seed_trace_key": key}, limit=1)), None):
            print(f"{DATASET_NAME} already has an example from trace {key}. Nothing was added.")
        else:
            client.create_examples(dataset_name=DATASET_NAME, examples=[example])
            print(f"Added 1 example to {DATASET_NAME}.")
    except Exception as exc:  # noqa: BLE001 - the example above stays printed; the UI steps of §5 do the same
        print(f"⏭️  The dataset step failed ({type(exc).__name__}). Add the example with the UI steps above instead.")
else:
    print(f"⏭️  Not sent. With a key and CREATE_DATASET = True, this cell adds the example to {DATASET_NAME}.")

## 6. Run a real report (in the UI)

With Plus or Enterprise and a provider key in the workspace, you can run a report on the seeded day yourself. It can take up to 30 minutes, so start it at the beginning of a session and read it at the end. The report's model calls are billed to the workspace's provider key (§1 gives the cost per 1,000 threads).

| Step | In the UI |
|---|---|
| 1. Open your TechHub traces | **Tracing** > agent **TechHub support** > **Production**; in a project-based workspace, **Tracing** > project `techhub-deep-agent-production`. Set the time range to at least the last 2 days |
| 2. Start a report | **+ New** in the top right corner > **New Insights Report**, then a name |
| 3. Model | If Insights has no model yet, a Workspace Admin adds the provider key under **Settings > Integrations > Provider secrets** and picks the models for Insights under the model configurations. The settings pages were reorganised in November 2026, so check these labels in your workspace |
| 4. Guided questions | With **Auto** on, answer the questions about what the agent does, what you want to learn and how the traces are structured. For TechHub: the customer's message is the last item of `inputs.messages`, and a conversation has one or two turns. The exact question labels are UNVERIFIED |
| 5. Run | **Run job** |
| 6. Read it in this notebook | Open the finished report from the **Insights** tab. Copy the `clusterJobId` value from its URL into `.env` as `M7_INSIGHTS_REPORT_ID`, restart the kernel and run this notebook again |

This notebook does not call `client.generate_insights`, the SDK method that starts a report. It takes `chat_histories` (lists of messages) instead of an existing project, uploads them as traces into a new project, and, if you pass a model key and the workspace has none, stores your key as a workspace secret that the evaluators and the Playground of that workspace can then use. To report on an existing project, use the UI, the LangSmith CLI (`langsmith insights create --project <name> --config insights.json`) or the REST endpoint `POST /api/v1/sessions/{session_id}/insights`.

### What you did in code, and where it is in LangSmith

| Step in this notebook | Same step in the UI |
|---|---|
| `InsightsReportResult(**load_insights_report(seed))`, or `client.get_insights_report(...)` with your report id | The **Insights** tab of the project, then the report |
| `report.report.key_points`, `report.report.highlighted_traces` | The executive summary, with numbered links to traces |
| Clusters with `parent_id is None`, then the clusters under each | The category bars, then a click into a category |
| `apply_evaluator(...)` on one subcategory's conversations | The subcategory's traces table, read trace by trace |
| The example of §5, sent with `create_examples(...)` | **Add to Annotation Queue**, review, **Add to Dataset** |

## Recap

- Insights summarizes each conversation, groups the summaries into categories and subcategories, and reports error rate, latency and cost for each group. A report needs Plus or Enterprise, a provider key in the workspace and up to 30 minutes.
- In the sample report the largest category is Order status and tracking (78 conversations, 39%), and Where is my order has the highest error rate (19.6%), from the order-service outage.
- Silent failures show an error rate of 0.0%. Return window questions has 27 conversations, and the Section 1 evaluator flags 11 of them, all on 1.5.0.
- Categories and key points are model output. Check a claim against the traces, then send the traces to an annotation queue, review them, and keep the corrected ones as dataset examples.

**Next:** Section 3 reads an Engine issue board for the same day. The wrong return window is its issue 1.